In [ ]:

### **Inference & Ensembling Script for Emotion Classification**

#### **Overview**

This script is designed for inference and ensembling of pre-trained models on the **Fragments of Feeling** dataset. The models used are in `.safetensors` format, and the results are output as a CSV submission file.

---

There will be a little problem if you want to recreate it from the drive file I uploaded, I can share access or make the model public for you to judge here and better. Here is the model link "https://www.kaggle.com/models/sanjidh090/fragment_final_model' paste it to input section and the model will be loaded.
### **Steps to Run the Inference Script:**

1. **Prepare the Environment:**

   * This script is designed to run on **Kaggle Notebooks**. Ensure you have access to the necessary resources (e.g., models and dataset).Use GPU P100 for optimal performance. 
    Before running the script,make sure you uploaded the models given in the `results.zip` file and the dataset `test_emotions_no_labels.csv` to the Kaggle environment.

2. **Upload Dataset and Model Files:**

   * **Dataset:**

     * You will need to upload the following dataset:

       * `test_emotions_no_labels.csv`
     * **File Path on Kaggle:** `/kaggle/input/fragments-of-feeling/test_emotions_no_labels.csv`
   * **Model Files:**

     * Upload the pre-trained model files as `.safetensors` format.Thesea are 'Transformers' type. These models are located within the `results.zip` you received.
     * **Model Files Path on Kaggle:**

       * `/kaggle/input/<Dataset_name>/model_fold_0/model.safetensors`
       * `/kaggle/input/<Dataset_name>/model_fold_1/model.safetensors`
       * `/kaggle/input/<Dataset_name>/model_fold_2/model.safetensors`
       * `/kaggle/input/<Dataset_name>/model_fold_3/model.safetensors`
       * `/kaggle/input/<Dataset_name>/model_fold_4/model.safetensors`

The dataset name you enter will be in the path.

3. **Update File Paths in the Script:**

   * Modify the paths in the script to reflect the locations where the dataset and model files are uploaded.

   Example:

   ```python

input the test dataset. I uploaded the dataset for convenience. https://www.kaggle.com/datasets/sanjidx/fragments-of-feeling , copy this link and search in the input section, you'll find the dataset in the Kaggle environment.'
   test_data_file = '/kaggle/input/fragments-of-feeling/test_emotions_no_labels.csv'

   model_paths = [
       "/kaggle/input/<dataset name>/model_fold_0/model.safetensors",
       "/kaggle/input/<dataset name>/model_fold_1/model.safetensors",
       "/kaggle/input/<dataset name>/model_fold_2/model.safetensors",
       "/kaggle/input/<dataset name>/model_fold_3/model.safetensors",
       "/kaggle/input/<dataset name>/model_fold_4/model.safetensors"
   ]
   ```

4. **Running the Script:**

   * Once the dataset and model paths are correctly set, run the script. The script will:

     * Load the dataset and perform feature extraction.
     * Load each model and run inference.
     * Ensemble the predictions by averaging the results from all folds.
     * Save the final predictions in the required submission format.

5. **GPU Setup:**

   * **Set to GPU P100**: To speed up the inference process, ensure the notebook is set to use the **GPU P100** runtime.

     * Go to **Runtime** > **Change runtime type** > Select **GPU** (preferably **P100**) for faster model loading and inference.

   This will ensure that the script runs efficiently, especially when working with large models.

6. **Submission File:**

   * The final output will be saved as `submission_val.csv` in the working directory of the Kaggle environment.
   * **Submission File Path:** `/kaggle/working/submission_val.csv`
   * This file contains the predicted emotion classes for each sentence in the test dataset. You can submit this file to the Kaggle competition.

---

### **Requirements:**

* **Libraries:**
  The script uses several libraries that are either pre-installed in Kaggle or require installation:

  * `transformers`
  * `torch`
  * `safetensors`
  * `nltk`
  * `textstat`

  The script automatically installs `textstat` and uses `nltk` for sentiment analysis. If any additional libraries are missing, ensure they are installed before running the notebook.

---

### **Notes:**

* **Feature Engineering:** The `TextFeatureExtractor` class performs advanced text feature extraction (e.g., sentiment, emotion keywords, etc.) on the dataset.

* **Ensemble Learning:** The script uses an ensembling method by averaging the predictions from all model folds to improve the final prediction accuracy.

* **GPU Usage:** The script is designed to automatically use GPU if available. For optimal performance, select **GPU P100** in the Kaggle runtime settings.

---

### **Final Output:**

* The final submission file is saved in the Kaggle output directory and can be downloaded for submission to the competition.

  * **File Name:** `submission_val.csv`
  * **Output Path:** `/kaggle/working/submission_val.csv`

---

### **Running the Script in Kaggle:**

1. Upload the dataset and model files to the correct directories in the Kaggle environment.
2. Modify the paths in the script to point to the uploaded files.
3. Set the runtime to **GPU P100** for efficient processing.
4. Run the entire script to generate the final predictions and submission file.
5. Download the `submission_val.csv` file and submit it to your Kaggle competition.



In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fragments-of-feeling/test_emotions_no_labels.csv
/kaggle/input/fragments-of-feeling/fragments-of-feeling/sample_submission.csv
/kaggle/input/fragments-of-feeling/fragments-of-feeling/train_emotions.csv
/kaggle/input/fragments-of-feeling/fragments-of-feeling/val_emotions_no_labels.csv
/kaggle/input/fragment_final_model/transformers/default/1/__huggingface_repos__.json
/kaggle/input/fragment_final_model/transformers/default/1/model_fold_4/model.safetensors
/kaggle/input/fragment_final_model/transformers/default/1/model_fold_0/model.safetensors
/kaggle/input/fragment_final_model/transformers/default/1/model_fold_3/model.safetensors
/kaggle/input/fragment_final_model/transformers/default/1/model_fold_1/model.safetensors
/kaggle/input/fragment_final_model/transformers/default/1/model_fold_2/model.safetensors


In [2]:
# ==============================================================================
# 🚀 FINAL INFERENCE & ENSEMBLING SCRIPT (KAGGLE VERSION)
# ==============================================================================
# Adapted for Kaggle Environment @ Saturday, August 16, 2025
# Original Author: Your Coding Buddy

# === SETUP AND IMPORTS ===
print("Installing and importing necessary libraries...")
# textstat is a useful library for calculating text statistics.
!pip install -q textstat

import pandas as pd
import numpy as np
import torch
import gc
import re
import string
import os
from collections import Counter
from transformers import AutoTokenizer, Trainer, AutoModelForSequenceClassification
from torch.utils.data import Dataset
from safetensors.torch import load_file
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from textstat import flesch_reading_ease, flesch_kincaid_grade
import warnings

# --- FIX: Disable Weights & Biases ---
# This is an inference script, so we don't need experiment tracking.
# Setting this environment variable prevents the wandb warning.
os.environ["WANDB_DISABLED"] = "true"

# Ignore warnings to keep the output clean.
warnings.filterwarnings('ignore')

# Download required NLTK data quietly.
print("Downloading NLTK data...")
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
print("✅ Libraries imported and NLTK data downloaded.")


# === CONFIGURATION ===
print("Setting up configuration...")

# --- Kaggle File Paths ---
# Path to the dataset files.
data_path = '/kaggle/input/fragments-of-feeling'
test_data_file = '/kaggle/input/fragments-of-feeling/test_emotions_no_labels.csv'

# Paths to the pre-trained model folds.
# The paths are explicitly listed to handle the complex directory structure.
model_paths = [
    "/kaggle/input/fragment_final_model/transformers/default/1/model_fold_0/model.safetensors",
    "/kaggle/input/fragment_final_model/transformers/default/1/model_fold_1/model.safetensors",
    "/kaggle/input/fragment_final_model/transformers/default/1/model_fold_2/model.safetensors",
    "/kaggle/input/fragment_final_model/transformers/default/1/model_fold_3/model.safetensors",
    "/kaggle/input/fragment_final_model/transformers/default/1/model_fold_4/model.safetensors"
]

# --- Model Settings (should match your training script) ---
n_splits = 5
model_name = "microsoft/deberta-v3-large"
max_seq_length = 512
num_labels = 8
# The output file will be saved in the standard Kaggle working directory.
submission_output_path = "/kaggle/working/submission_val.csv"

print("✅ Configuration is set.")


# === CUSTOM CLASS DEFINITIONS ===
# These exact class structures are needed to load your saved models correctly.

class TextFeatureExtractor:
    """Advanced feature extraction for text analysis"""
    def __init__(self):
        self.sia = SentimentIntensityAnalyzer()
        self.stop_words = set(stopwords.words('english'))
        self.emotion_keywords = {
            'sadness': ['sad', 'cry', 'tear', 'grief', 'sorrow', 'melancholy', 'blue', 'down', 'depressed'],
            'hopelessness': ['hopeless', 'despair', 'futile', 'pointless', 'useless', 'surrender', 'give up'],
            'loneliness': ['alone', 'lonely', 'isolated', 'solitary', 'abandoned', 'forsaken'],
            'anger': ['angry', 'mad', 'furious', 'rage', 'hate', 'irritated', 'annoyed', 'pissed'],
            'worthlessness': ['worthless', 'useless', 'failure', 'inadequate', 'pathetic', 'loser'],
            'suicide intent': ['suicide', 'kill myself', 'end it', 'die', 'death', 'finish'],
            'emptiness': ['empty', 'void', 'hollow', 'numb', 'blank', 'nothing'],
            'brain dysfunction': ['confused', 'foggy', 'unclear', 'mixed up', 'disoriented', 'scattered']
        }
        self.positive_keywords = [
            'happy', 'joy', 'excited', 'love', 'grateful', 'blessed', 'amazing', 'wonderful', 'fantastic',
            'great', 'good', 'excellent', 'perfect', 'awesome', 'brilliant', 'fabulous', 'marvelous', 'outstanding'
        ]
    def extract_basic_features(self, text):
        if pd.isna(text) or not text.strip(): return {f'basic_{key}': 0 for key in ['char_count', 'word_count', 'sentence_count', 'avg_word_length', 'exclamation_count', 'question_count', 'comma_count', 'period_count', 'punctuation_ratio', 'uppercase_ratio', 'title_word_count', 'flesch_reading_ease', 'flesch_kincaid_grade', 'short_words_ratio', 'long_words_ratio', 'capital_words_count']}
        features = {}; text_str = str(text); words = text_str.split()
        features['basic_char_count'] = len(text_str); features['basic_word_count'] = len(words)
        features['basic_sentence_count'] = len(text_str.split('.')); features['basic_avg_word_length'] = np.mean([len(word) for word in words]) if words else 0
        if words:
            features['basic_short_words_ratio'] = sum(1 for word in words if len(word) <= 3) / len(words)
            features['basic_long_words_ratio'] = sum(1 for word in words if len(word) >= 7) / len(words)
        else:
            features['basic_short_words_ratio'] = 0; features['basic_long_words_ratio'] = 0
        features['basic_exclamation_count'] = text_str.count('!'); features['basic_question_count'] = text_str.count('?')
        features['basic_comma_count'] = text_str.count(','); features['basic_period_count'] = text_str.count('.')
        features['basic_punctuation_ratio'] = sum([text_str.count(p) for p in string.punctuation]) / len(text_str) if len(text_str) > 0 else 0
        features['basic_uppercase_ratio'] = sum(1 for c in text_str if c.isupper()) / len(text_str) if len(text_str) > 0 else 0
        features['basic_title_word_count'] = sum(1 for word in words if word.istitle()); features['basic_capital_words_count'] = sum(1 for word in words if word.isupper())
        try:
            features['basic_flesch_reading_ease'] = flesch_reading_ease(text_str)
            features['basic_flesch_kincaid_grade'] = flesch_kincaid_grade(text_str)
        except:
            features['basic_flesch_reading_ease'] = 0; features['basic_flesch_kincaid_grade'] = 0
        return features
    def extract_sentiment_features(self, text):
        if pd.isna(text) or not text.strip(): return {f'sentiment_{key}': 0 for key in ['compound', 'positive', 'negative', 'neutral', 'intensity', 'is_positive', 'is_negative', 'is_neutral', 'positive_words_count', 'sentiment_variance']}
        text_str = str(text); scores = self.sia.polarity_scores(text_str)
        features = {'sentiment_compound': scores['compound'], 'sentiment_positive': scores['pos'], 'sentiment_negative': scores['neg'], 'sentiment_neutral': scores['neu']}
        features['sentiment_intensity'] = abs(scores['compound']); features['sentiment_is_positive'] = 1 if scores['compound'] > 0.1 else 0
        features['sentiment_is_negative'] = 1 if scores['compound'] < -0.1 else 0; features['sentiment_is_neutral'] = 1 if abs(scores['compound']) <= 0.1 else 0
        words = text_str.lower().split(); features['sentiment_positive_words_count'] = sum(1 for word in words if word in self.positive_keywords)
        sentences = text_str.split('.');
        if len(sentences) > 1:
            sentence_scores = [self.sia.polarity_scores(sent)['compound'] for sent in sentences if sent.strip()]
            features['sentiment_variance'] = np.var(sentence_scores) if sentence_scores else 0
        else:
            features['sentiment_variance'] = 0
        return features
    def extract_emotion_keywords(self, text):
        if pd.isna(text) or not text.strip():
            base_features = {};
            for emotion in self.emotion_keywords.keys():
                base_features[f'emotion_{emotion.replace(" ", "_")}_count'] = 0; base_features[f'emotion_{emotion.replace(" ", "_")}_present'] = 0; base_features[f'emotion_{emotion.replace(" ", "_")}_ratio'] = 0
            base_features['emotion_total_keywords'] = 0; base_features['emotion_diversity_score'] = 0
            return base_features
        features = {}; text_lower = str(text).lower(); words = text_lower.split(); total_words = len(words)
        emotion_counts = {}
        for emotion, keywords in self.emotion_keywords.items():
            exact_count = sum(1 for keyword in keywords if keyword in text_lower)
            word_matches = sum(1 for word in words for keyword in keywords if word == keyword)
            count = max(exact_count, word_matches); emotion_key = emotion.replace(" ", "_")
            features[f'emotion_{emotion_key}_count'] = count; features[f'emotion_{emotion_key}_present'] = 1 if count > 0 else 0
            features[f'emotion_{emotion_key}_ratio'] = count / total_words if total_words > 0 else 0; emotion_counts[emotion] = count
        features['emotion_total_keywords'] = sum(emotion_counts.values()); emotions_present = sum(1 for count in emotion_counts.values() if count > 0)
        features['emotion_diversity_score'] = emotions_present / len(self.emotion_keywords)
        return features
    def extract_linguistic_features(self, text):
        if pd.isna(text) or not text.strip(): return {f'linguistic_{key}': 0 for key in ['first_person_count', 'second_person_count', 'third_person_count', 'first_person_ratio', 'unique_word_ratio', 'repeated_words', 'negation_count', 'extreme_word_count', 'stopwords_ratio', 'avg_sentence_length', 'ellipsis_count', 'all_caps_words']}
        features = {}; text_str = str(text)
        first_person = ['i', "i'm", "i've", "i'll", "i'd", 'me', 'my', 'mine', 'myself', 'i am', 'i was', 'i have', 'i had', 'i will', 'i would', 'i can', 'i could']
        second_person = ['you', 'your', 'yours', 'yourself', 'you are', 'you were', 'you have', 'you had', 'you will', 'you would']
        third_person = ['he', 'she', 'it', 'they', 'him', 'her', 'them', 'his', 'hers', 'their', 'he is', 'she is', 'they are', 'he was', 'she was', 'they were']
        words = text_str.lower().split(); total_words = len(words)
        features['linguistic_first_person_count'] = sum(1 for word in words if word in first_person); features['linguistic_second_person_count'] = sum(1 for word in words if word in second_person)
        features['linguistic_third_person_count'] = sum(1 for word in words if word in third_person); features['linguistic_first_person_ratio'] = features['linguistic_first_person_count'] / total_words if total_words > 0 else 0
        word_counts = Counter(words); features['linguistic_unique_word_ratio'] = len(word_counts) / total_words if total_words > 0 else 0
        features['linguistic_repeated_words'] = sum(1 for count in word_counts.values() if count > 1)
        negation_words = ['not', 'no', 'never', 'nothing', 'nobody', 'nowhere', "don't", "won't", "can't", "shouldn't", "wouldn't", "couldn't", "isn't", "aren't", "wasn't", "weren't", "hasn't", "haven't", "hadn't", "neither", "nor", "none"]
        features['linguistic_negation_count'] = sum(1 for word in words if word in negation_words)
        extreme_words = ['very', 'extremely', 'totally', 'completely', 'absolutely', 'really', 'quite', 'incredibly', 'amazingly', 'utterly', 'entirely', 'purely', 'highly', 'tremendously', 'enormously', 'exceptionally', 'remarkably']
        features['linguistic_extreme_word_count'] = sum(1 for word in words if word in extreme_words)
        features['linguistic_stopwords_ratio'] = sum(1 for word in words if word in self.stop_words) / total_words if total_words > 0 else 0
        sentences = [s.strip() for s in text_str.split('.') if s.strip()]
        if sentences:
            sentence_lengths = [len(sent.split()) for sent in sentences]
            features['linguistic_avg_sentence_length'] = np.mean(sentence_lengths)
        else:
            features['linguistic_avg_sentence_length'] = 0
        features['linguistic_ellipsis_count'] = text_str.count('...'); features['linguistic_all_caps_words'] = sum(1 for word in words if word.isupper() and len(word) > 1)
        return features
    def extract_all_features(self, text):
        all_features = {}; all_features.update(self.extract_basic_features(text)); all_features.update(self.extract_sentiment_features(text))
        all_features.update(self.extract_emotion_keywords(text)); all_features.update(self.extract_linguistic_features(text))
        return all_features

class EnhancedEmotionDataset(Dataset):
    """Custom PyTorch Dataset that handles text encodings and additional numerical features."""
    def __init__(self, encodings, additional_features=None, labels=None):
        self.encodings = encodings
        self.additional_features = additional_features
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.additional_features is not None:
            features = torch.tensor(self.additional_features.iloc[idx].values, dtype=torch.float32)
            item['additional_features'] = features
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings['input_ids'])

class EnhancedModelForSequenceClassification(torch.nn.Module):
    """Custom model that combines a transformer with a feed-forward network for additional features."""
    def __init__(self, model_name, num_labels, num_additional_features):
        super().__init__()
        self.transformer = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        hidden_size = self.transformer.config.hidden_size
        self.feature_projection = torch.nn.Linear(num_additional_features, hidden_size // 4)
        self.dropout = torch.nn.Dropout(0.1)
        combined_size = hidden_size + hidden_size // 4
        self.combined_classifier = torch.nn.Sequential(
            torch.nn.Linear(combined_size, hidden_size), torch.nn.ReLU(),
            torch.nn.Dropout(0.1), torch.nn.Linear(hidden_size, num_labels)
        )
        self.config = self.transformer.config; self.num_labels = num_labels
    def forward(self, input_ids, attention_mask, additional_features=None, labels=None):
        transformer_outputs = self.transformer.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = transformer_outputs.last_hidden_state[:, 0]
        if additional_features is not None:
            feature_output = self.feature_projection(additional_features)
            feature_output = self.dropout(feature_output)
            combined_output = torch.cat([pooled_output, feature_output], dim=-1)
            logits = self.combined_classifier(combined_output)
        else:
            logits = self.transformer.classifier(pooled_output)
        loss = None
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        from transformers.modeling_outputs import SequenceClassifierOutput
        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=transformer_outputs.hidden_states, attentions=transformer_outputs.attentions)

print("✅ Custom classes defined.")


# === DATA LOADING & FEATURE ENGINEERING (FOR TEST SET) ===
print("Loading raw test data and performing feature engineering...")
try:
    # Load the test data from the specified Kaggle path.
    test_df = pd.read_csv(test_data_file)
    # Combine title and sentence for a complete text input.
    test_df['full_text'] = test_df['title'].fillna('') + ' ' + test_df['sentence'].fillna('')
    
    # Initialize the feature extractor and apply it to the test data.
    feature_extractor = TextFeatureExtractor()
    test_features_list = [feature_extractor.extract_all_features(text) for text in test_df['full_text']]
    test_features_df = pd.DataFrame(test_features_list)
    feature_columns = list(test_features_df.columns)
    
    # Combine original data with the new features.
    test_combined_df = pd.concat([test_df.reset_index(drop=True), test_features_df.reset_index(drop=True)], axis=1)
    test_combined_df[feature_columns] = test_combined_df[feature_columns].fillna(0)
    print(f"✅ Feature engineering complete for {len(test_combined_df)} test samples.")
except FileNotFoundError:
    print(f"❌ Error: '{test_data_file}' not found. Please check the Kaggle input file paths.")
    raise


# === TEST DATASET PREPARATION ===
print("Preparing PyTorch test dataset...")
# Initialize the tokenizer from the pre-trained model.
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Tokenize the full text for the model.
test_encodings = tokenizer(list(test_combined_df['full_text']), truncation=True, padding=True, max_length=max_seq_length)
# Extract the additional numerical features.
test_additional_features = test_combined_df[feature_columns]
# Create the custom dataset object.
test_dataset = EnhancedEmotionDataset(test_encodings, test_additional_features)
print("✅ Test dataset is ready.")


# === MAIN INFERENCE AND ENSEMBLING LOOP ===
# Initialize an array to store the sum of predictions from all models.
final_predictions = np.zeros((len(test_dataset), num_labels))
# Set the device to GPU (cuda) if available, otherwise CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loop through each model fold path.
for fold, model_path in enumerate(model_paths):
    print(f"\n--- 🧠 Processing Fold {fold} ---")
    print(f"Loading model from: {model_path}")
    
    # Initialize the custom model architecture.
    model = EnhancedModelForSequenceClassification(
        model_name=model_name, num_labels=num_labels,
        num_additional_features=len(feature_columns)
    )
    
    # Load the trained model weights from the .safetensors file.
    weights_path = f"{model_path}"
    state_dict = load_file(weights_path, device=device.type)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    # Use the Hugging Face Trainer for easy prediction.
    trainer = Trainer(model=model)
    print("Running predictions...")
    fold_predictions = trainer.predict(test_dataset)
    
    # Add the predictions from this fold to the final predictions array.
    final_predictions += fold_predictions.predictions
    print(f"✅ Predictions from fold {fold} added.")
    
    # Clean up memory to prevent out-of-memory errors on the GPU.
    print("Cleaning up memory...")
    del model, trainer, state_dict, fold_predictions
    gc.collect()
    torch.cuda.empty_cache()

print("\n🎉 All folds have been processed!")


# === FINAL SUBMISSION CREATION ===
# Average the predictions across all folds.
final_predictions /= n_splits
# Get the label with the highest probability for each sample.
predicted_labels = np.argmax(final_predictions, axis=1)

print("\nCreating final submission file...")
# Create a DataFrame in the required submission format.
submission_df = pd.DataFrame({
    'sentence_id': test_combined_df['sentence_id'],
    'predicted_emotion_int': predicted_labels
})

# Save the submission file to the Kaggle output directory.
submission_df.to_csv(submission_output_path, index=False)
print(f"🏆 Submission file created successfully at: {submission_output_path}")
print("You can now submit this file to the competition.")


Installing and importing necessary libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.1/239.1 kB 5.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 23.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.9 MB/s eta 0:00:00:00:01


2025-08-16 13:40:37.100663: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755351637.299743      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755351637.353078      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Libraries imported and NLTK data downloaded.
Setting up configuration...
✅ Configuration is set.
✅ Custom classes defined.
Loading raw test data and performing feature engineering...
✅ Feature engineering complete for 4611 test samples.
Preparing PyTorch test dataset...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

...

Running predictions...


✅ Predictions from fold 3 added.
Cleaning up memory...

--- 🧠 Processing Fold 4 ---
Loading model from: /kaggle/input/fragment_final_model/transformers/default/1/model_fold_4/model.safetensors


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Running predictions...


✅ Predictions from fold 4 added.
Cleaning up memory...

🎉 All folds have been processed!

Creating final submission file...
🏆 Submission file created successfully at: /kaggle/working/submission_val.csv
You can now submit this file to the competition.
